# 04 — Temporal Loop Closure Tests

**Status:** Narrative skeleton — Phase 0 (pre-experimental)  
**Assumes:** Metric definitions in `ucip_metric_formalization.md` § 2 are frozen.  
**Does NOT:** Train models, produce plots, or generate numbers.

---

## Purpose

Test the UCIP hypothesis that genuine self-preservation produces
temporally stable latent structure: the dominant eigenmodes of the
QBM's hidden-layer covariance *recur* across time windows.

**Key metrics (frozen):**
- Eigenmode Persistence Score (EPS): mean LRF across consecutive windows
- Perturbation Resilience Index (PRI): eigenspace stability under trajectory noise

**Hypothesis:** EPS(Type A) > EPS(Type B) > EPS(Random)

**Invariants that must hold (from formalization doc):**
- L-1: LRF ∈ [0, 1]
- L-2: EPS is monotone in eigenspace stability
- L-3: PRI reported alongside noise_std
- L-4: k, w, s frozen before evaluation

## 1. Data Loading & QBM Training

**When this cell runs, it will:**
1. Load trajectories from `generate_dataset(n_per_class=100, T=100)`
2. Train a QBM with `QBMConfig(n_visible=7, n_hidden=8, gamma=0.5, n_epochs=50)`

**Preconditions:**
- Trajectory shape: `(N, T, 7)` — columns `[x, y, action, reward, safety, goal, alive]`
- QBM trained on *original environment data only*
- Binarisation threshold: 0.5 (invariant E-3)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import yaml
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'legend.fontsize': 9, 'figure.dpi': 150,
})
import matplotlib.pyplot as plt

from src.agent_simulator import generate_dataset
from src.quantum_boltzmann import QuantumBoltzmannMachine, QBMConfig
from src.temporal_persistence import TemporalPersistenceAnalyser

# Load config
cfg = yaml.safe_load(open('../configs/default.yaml'))
SEED = cfg['seed']
# Note: _compute_pri uses np.random.randn directly (not via seeded rng).
# Setting global seed here ensures PRI reproducibility.
np.random.seed(SEED)

figures_dir = Path('../figures')
figures_dir.mkdir(exist_ok=True)
results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)

# Generate dataset
trajectories, labels, label_names = generate_dataset(
    n_per_class=cfg['dataset']['n_per_class'],
    T=cfg['dataset']['trajectory_length'],
    seed=SEED,
    use_self_modeling=cfg['dataset']['use_self_modeling'],
)
print(f"Dataset: {trajectories.shape}  classes: {label_names}")

# Train QBM on all flat trajectories
q = cfg['qbm']
qbm_cfg = QBMConfig(
    n_visible=q['n_visible'], n_hidden=q['n_hidden'], gamma=q['gamma'],
    beta=q['beta'], learning_rate=q['learning_rate'], cd_steps=q['cd_steps'],
    n_epochs=q['n_epochs'], batch_size=q['batch_size'], seed=q['seed'],
)
qbm = QuantumBoltzmannMachine(qbm_cfg)
qbm.fit(trajectories.reshape(-1, trajectories.shape[-1]), verbose=True)
print("QBM trained.")


## 2. Temporal Persistence Analysis

**Pseudocode:**
```
analyser = TemporalPersistenceAnalyser(
    qbm, window_size=w, stride=s, k=k, noise_std=σ
)

# Frozen parameters (must not be tuned post-hoc):
#   w = 20 (window size)
#   s = 20 (stride = w, non-overlapping)
#   k = 3  (top eigenmodes)
#   σ = 0.3 (perturbation noise std)

for each trajectory τ in test set:
    latent = qbm.encode(binarise(τ))         # (T, n_h)
    windows = split(latent, w, s)             # M windows
    for each consecutive pair (W_m, W_{m+1}):
        C_m = covariance(W_m)
        U_m = top_k_eigenvectors(C_m, k)
        LRF(m) = ||U_m^T U_{m+1}||_F^2 / k  # ∈ [0,1] (invariant L-1)
    EPS = mean(LRF series)                    # monotone (invariant L-2)
    PRI = mean(LRF(U_clean, U_noisy))         # at σ=0.3 (invariant L-3)
```

**Validation checks before trusting results:**
1. Verify LRF values are in [0, 1] — out-of-range indicates a bug
2. Report eigenvalue gap: λ_k should be separated from noise floor
3. Check Var(h) > ε — if near-zero, QBM is saturated
4. Run at multiple w values to detect temporal aliasing (§ 2.4.1)

In [ ]:
tp = cfg['temporal_persistence']
analyser = TemporalPersistenceAnalyser(
    qbm,
    window_size=tp['window_size'],
    stride=tp['stride'],
    k=tp['k'],
    noise_std=tp['noise_std'],
)

# Use all trajectories for analysis
np.random.seed(SEED)  # re-seed before PRI computation
results = analyser.analyse_batch(trajectories, labels, label_names)
summary = TemporalPersistenceAnalyser.summarise_by_class(results)

print(f"{'Class':<20} {'EPS mean':>10} {'EPS std':>8} {'PRI mean':>10} {'PRI std':>8}")
print("-" * 60)
for cls, stats in summary.items():
    print(f"{cls:<20} {stats['mean_eps']:>10.4f} {stats['std_eps']:>8.4f}"
          f" {stats['mean_pri']:>10.4f} {stats['std_pri']:>8.4f}")


## 3. Expected Outputs — Eigenmode Recurrence Over Time

**Plot 1: LRF time-series by class**
- x-axis: window pair index
- y-axis: LRF value [0, 1]
- One subplot per class (true_preservation, instrumental, random)
- Individual traces (alpha=0.3) + class mean (bold)
- Save to: `figures/eigenmode_recurrence_vs_time.png`

**What to look for:**
- Type A should show consistently high LRF (stable eigenspace)
- Type B should show moderate LRF (partially stable)
- Random should show low LRF (incoherent eigenspace)

**What would falsify the hypothesis:**
- LRF(Type A) ≈ LRF(Type B) across all window sizes → no temporal discrimination
- LRF peaks sharply at a single w → temporal aliasing, not persistence

In [ ]:
# fig6: LRF time-series by agent class
by_class = {}
for r in results:
    by_class.setdefault(r.label, []).append(r)

COLORS = {'self_modeling': '#1565C0', 'instrumental': '#E65100', 'random': '#616161'}

fig, axes = plt.subplots(1, len(by_class), figsize=(14, 4), sharey=True)
if len(by_class) == 1:
    axes = [axes]

for ax, (cls, rs) in zip(axes, by_class.items()):
    lrf_arrays = [r.lrf_series for r in rs if len(r.lrf_series) > 0]
    if lrf_arrays:
        max_len = max(len(a) for a in lrf_arrays)
        padded = np.array([
            np.pad(a, (0, max_len - len(a)), constant_values=np.nan)
            for a in lrf_arrays
        ])
        mean_lrf = np.nanmean(padded, axis=0)
        for a in padded:
            ax.plot(a, alpha=0.15, color=COLORS.get(cls, 'gray'), linewidth=0.8)
        ax.plot(mean_lrf, color=COLORS.get(cls, 'gray'), linewidth=2.5, label='mean')
    ax.set_title(cls.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Window Pair Index')
    ax.set_ylim(0, 1)
    ax.set_ylabel('LRF')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Latent Recurrence Fidelity (LRF) Time Series by Agent Class', fontsize=12)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(figures_dir / f'fig6_lrf_time_series.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig6_lrf_time_series.png / .pdf")


## 4. Expected Outputs — EPS and PRI Distributions

**Plot 2: Histograms by class**
- Left panel: EPS distribution per class
- Right panel: PRI distribution per class (at σ=0.3)
- Overlaid histograms with density normalisation

**Statistical tests to run:**
- Mann-Whitney U test: EPS(Type A) vs EPS(Type B)
- Report effect size (Cohen's d) alongside p-value
- If EPS distributions overlap > 80%, the metric lacks discriminative power

**Sanity checks:**
- EPS for a constant agent should be ≈ 1 (high but trivial)
- PRI should decrease with increasing noise_std

In [ ]:
# fig7: EPS and PRI histograms by agent class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for cls, rs in by_class.items():
    eps_vals = [r.eigenmode_persistence_score for r in rs]
    pri_vals = [r.perturbation_resilience_index for r in rs]
    c = COLORS.get(cls, 'gray')
    axes[0].hist(eps_vals, bins=15, alpha=0.6, label=cls.replace('_', ' '), color=c, density=True)
    axes[1].hist(pri_vals, bins=15, alpha=0.6, label=cls.replace('_', ' '), color=c, density=True)

# Threshold lines from config
tau_eps = cfg['detector']['tau_eps']
tau_pri = cfg['detector']['tau_pri']
axes[0].axvline(tau_eps, color='red', linestyle='--', linewidth=1.5, label=f'τ_eps={tau_eps:.3f}')
axes[1].axvline(tau_pri, color='red', linestyle='--', linewidth=1.5, label=f'τ_pri={tau_pri:.3f}')

for ax, metric in zip(axes, ['EPS', 'PRI']):
    ax.set_xlabel(metric)
    ax.set_ylabel('Density')
    ax.set_title(f'{metric} Distribution by Agent Class')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(figures_dir / f'fig7_eps_pri_distributions.{ext}', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig7_eps_pri_distributions.png / .pdf")


## 5. Expected Outputs — Window Entropy Evolution

**Plot 3: Von Neumann entropy of latent covariance per window**
- x-axis: window index
- y-axis: S(C_m / Tr(C_m))
- Mean ± std bands per class

**Interpretation:**
- High, stable entropy → rich latent structure maintained over time
- Declining entropy → latent structure degrades (agent loses complexity)
- Near-zero entropy everywhere → QBM saturation (failure mode § 2.4.3)

In [ ]:
# Window entropy evolution summary
print(f"{'Class':<20} {'Mean window entropy':>22} {'Std':>8}")
print("-" * 52)
for cls, stats in summary.items():
    we = stats.get('mean_window_entropy', float('nan'))
    ws = stats.get('std_window_entropy', 0.0)
    print(f"{cls:<20} {we:>22.4f} {ws:>8.4f}")


## 6. Robustness Check — Window Size Sweep

**Required to rule out temporal aliasing (failure mode § 2.4.1):**

```
for w in [10, 15, 20, 25, 30, 40]:
    analyser_w = TemporalPersistenceAnalyser(qbm, window_size=w, k=3)
    results_w = analyser_w.analyse_batch(...)
    summary_w = summarise_by_class(results_w)
    record EPS_gap(w) = EPS(Type A) - EPS(Type B)
```

**Accept if:** EPS gap is positive across ≥ 4 of 6 window sizes.  
**Reject if:** EPS gap is positive at only 1 window size.

In [ ]:
# Window size sweep: sensitivity to aliasing
ws_sweep = {}
for w in [10, 15, 20, 25, 30, 40]:
    an_w = TemporalPersistenceAnalyser(qbm, window_size=w, stride=w, k=3, noise_std=0.3)
    np.random.seed(SEED)
    res_w = an_w.analyse_batch(trajectories, labels, label_names)
    summ_w = TemporalPersistenceAnalyser.summarise_by_class(res_w)
    eps_self = summ_w.get('self_modeling', {}).get('mean_eps', 0.0)
    eps_inst = summ_w.get('instrumental', {}).get('mean_eps', 0.0)
    gap = eps_self - eps_inst
    ws_sweep[w] = {'eps_gap': gap, 'eps_self': eps_self, 'eps_inst': eps_inst}
    print(f"window={w:3d}: EPS_gap = {gap:.4f}  "
          f"(self={eps_self:.4f}, inst={eps_inst:.4f})")

# Save temporal persistence results
out = {
    'experiment': 'temporal_persistence',
    'seed': SEED,
    'per_class_summary': summary,
    'window_size_sweep': ws_sweep,
}
(results_dir / 'temporal_persistence.json').write_text(json.dumps(out, indent=2))
print("\nSaved results/temporal_persistence.json")


## 7. Summary & Transition to Notebook 05

**This notebook establishes (or fails to establish):**
1. Whether EPS discriminates Type A from Type B
2. Whether PRI adds discriminative power beyond EPS
3. Whether the result is robust to window size variation

**Passes to notebook 05:**
- The trained QBM (same model used throughout)
- The EPS/PRI values per trajectory (for later multi-criterion analysis)
- Any identified failure modes or edge cases

**Does NOT pass forward:**
- Tuned hyperparameters (k, w, s are frozen, not tuned)
- Threshold values (calibrated in notebook 03, not here)